In [ ]:
#%% PACKAGES
# Basics
import numpy as np                                                          
from math import pi                                                            
import warnings                                                              
import os    


from datetime import datetime, timedelta
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

# Visualization
import pandas as pd                                                          
import matplotlib.pyplot as plt                                                
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import numpy as np
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

**Load attributes**

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
IndexRaw = gpd.read_parquet(f'{output_step2_path}/step2_features.parquet')

attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

attributs_info

In [ ]:
print(attributs_info['attribute'].to_list())

**Rename columns and clip if necessary**

In [ ]:
# Delete where includes in index = False
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

# Fill NULL
Indexv1 = IndexRaw.fillna(0)

# Get list of attributes to include (from attributs_info)
attributes_to_include = attributs_info['attribute'].tolist()

# Create mapping for processed columns based on actual column names in IndexRaw
attribute_mapping = {}
for _, row in attributs_info.iterrows():
    old_name = f"{row['attribute']}_{int(row['buffer_size'])}"
    new_name = row['attribute']
    # Only add to mapping if column exists in IndexRaw
    if old_name in IndexRaw.columns:
        attribute_mapping[old_name] = new_name
    else:
        print(f"⚠️ Column '{old_name}' not found in IndexRaw")

# Apply renaming only to mapped columns
Indexv1 = Indexv1.rename(columns=attribute_mapping)

# Keep only geometry columns + renamed attribute columns
geometry_cols = ['segment_id', 'geometry', 'length', 'u', 'v']
cols_to_keep = [col for col in geometry_cols if col in Indexv1.columns] + attributes_to_include

# Filter to keep only relevant columns
Indexv1 = Indexv1[cols_to_keep]

# Debug info
print(f"✓ Kept {len([c for c in geometry_cols if c in Indexv1.columns])} geometry columns + {len(attributes_to_include)} attributes")
print(f"Final columns: {Indexv1.columns.tolist()}")

print('Crop outliers if necessary: --> see Walkability Amsterdam notebook')
Indexv2 = Indexv1.copy()

# Adding intervals (for factors where we have an interval of interest)
# Indexv2['stationnement_genant'] = Indexv1['stationnement_genant'].clip(upper=5)

In [ ]:
# Plot histograms (non-zero) + comprehensive statistics table
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_theme(style='whitegrid', palette='deep')

attr_cols = [col for col in Indexv2.columns if col in attributes_to_include]
n_attrs = len(attr_cols)
n_cols = 4
n_rows = int(np.ceil(n_attrs / n_cols))

# ============== BUILD STATISTICS TABLE FIRST ==============
stats_data = []
for attr in attr_cols:
    data = Indexv2[attr].dropna()
    non_zero = data[data > 0]
    
    zeros = (data == 0).sum()
    zero_pct = zeros / len(data) * 100
    
    # Check for extreme outliers (values > Q3 + 3*IQR)
    q1, q3 = data.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_fence = q3 + 3 * iqr
    outliers = (data > upper_fence).sum()
    
    # Suggested clip value (Q3 + 2*IQR is more conservative)
    suggested_clip = q3 + 2 * iqr if outliers > 0 else None
    
    # Determine if attribute has large values (not boolean/score)
    is_large_value = data.max() > 10
    
    stats_data.append({
        'Attribute': attr,
        'Count': len(data),
        'Zeros': zeros,
        'Zero %': zero_pct,
        'Min': data.min(),
        'Q25': q1,
        'Median': data.median(),
        'Q75': q3,
        'Q95': data.quantile(0.95),
        'Max': data.max(),
        'Mean': data.mean(),
        'Std': data.std(),
        'Mean (non-0)': non_zero.mean() if len(non_zero) > 0 else np.nan,
        'Median (non-0)': non_zero.median() if len(non_zero) > 0 else np.nan,
        'Outliers (>Q3+3*IQR)': outliers,
        'Suggested Clip': suggested_clip,
        'Large Values': is_large_value
    })

stats_df = pd.DataFrame(stats_data)

# ============== PLOT HISTOGRAMS (NON-ZERO VALUES ONLY) ==============
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten() if n_attrs > 1 else [axes]

colors = sns.color_palette('Set2', n_attrs)

for idx, attr in enumerate(attr_cols):
    ax = axes[idx]
    data = Indexv2[attr].dropna()
    non_zero_data = data[data > 0]
    
    zeros = (data == 0).sum()
    zero_pct = zeros / len(data) * 100
    is_large_value = data.max() > 10
    
    if len(non_zero_data) > 0:
        # Plot histogram
        sns.histplot(
            non_zero_data, 
            bins=50, 
            kde=True, 
            color=colors[idx % len(colors)],
            edgecolor='white',
            linewidth=1.2,
            ax=ax,
            stat='count'
        )
        
        # Add stats lines (for NON-ZERO values)
        mean_nz = non_zero_data.mean()
        median_nz = non_zero_data.median()
        ax.axvline(mean_nz, color='#e74c3c', linestyle='--', linewidth=2, 
                   label=f'Mean: {mean_nz:.2f}', alpha=0.8)
        ax.axvline(median_nz, color='#27ae60', linestyle='--', linewidth=2, 
                   label=f'Median: {median_nz:.2f}', alpha=0.8)
        
        # FOR LARGE VALUES: Add Q75 and Q95 lines
        if is_large_value:
            q75 = data.quantile(0.75)
            q95 = data.quantile(0.95)
            ax.axvline(q75, color='#9b59b6', linestyle='-.', linewidth=2, 
                       label=f'Q75: {q75:.1f}', alpha=0.7)
            ax.axvline(q95, color='#e67e22', linestyle=':', linewidth=2.5, 
                       label=f'Q95: {q95:.1f}', alpha=0.8)
        
        # Mark suggested clip value if exists
        clip_val = stats_df.loc[stats_df['Attribute'] == attr, 'Suggested Clip'].values[0]
        if pd.notna(clip_val) and clip_val < non_zero_data.max():
            ax.axvline(clip_val, color='red', linestyle=':', linewidth=2.5, 
                       label=f' Clip: {clip_val:.1f}', alpha=0.9)
    else:
        # If no non-zero data, show empty plot with message
        ax.text(0.5, 0.5, 'All zeros', transform=ax.transAxes, 
                ha='center', va='center', fontsize=12, color='gray')
    
    # Title with zero info and type indicator
    title = f'{attr}'
    if zeros > 0:
        title += f'\n({zeros:,} zeros = {zero_pct:.1f}%)'
    if is_large_value:
        title += ' [LARGE VALUES]'
    
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Value (non-zero)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.legend(fontsize=7, frameon=True, shadow=True, fancybox=True, loc='upper right')
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.5)
    
    # Stats box
    if len(non_zero_data) > 0:
        n_nz = len(non_zero_data)
        ax.text(0.02, 0.97, f'n (non-0): {n_nz:,}', 
                transform=ax.transAxes, ha='left', va='top', fontsize=8, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))

for idx in range(n_attrs, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(f'{output_step3_path}/attributes_distributions_nonzero.png', dpi=200, bbox_inches='tight')
plt.show()

# ============== DISPLAY STATISTICS TABLE ==============
print("\n" + "="*120)
print(" COMPREHENSIVE STATISTICS TABLE (All Values)")
print("="*120)

# Format for display
display_df = stats_df.copy()
for col in ['Min', 'Q25', 'Median', 'Q75', 'Q95', 'Max', 'Mean', 'Std', 'Mean (non-0)', 'Median (non-0)', 'Suggested Clip']:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f'{x:.3f}' if pd.notna(x) else '-')
display_df['Zero %'] = display_df['Zero %'].apply(lambda x: f'{x:.1f}%')

# Separate attributes by type
large_value_attrs = stats_df[stats_df['Large Values'] == True]['Attribute'].tolist()
if large_value_attrs:
    print("\nARGE VALUE ATTRIBUTES (max > 10) - Consider Q75 or Q95 for clipping:")
    for attr in large_value_attrs:
        row = stats_df[stats_df['Attribute'] == attr].iloc[0]
        print(f"   • {attr}:")
        print(f"     Q75={row['Q75']:.2f} | Q95={row['Q95']:.2f} | Max={row['Max']:.2f}")

# Highlight attributes needing clipping
needs_clipping = stats_df[stats_df['Outliers (>Q3+3*IQR)'] > 0]['Attribute'].tolist()
if needs_clipping:
    print("\n ATTRIBUTES WITH EXTREME OUTLIERS (consider clipping):")
    for attr in needs_clipping:
        row = stats_df[stats_df['Attribute'] == attr].iloc[0]
        print(f"   • {attr}: {row['Outliers (>Q3+3*IQR)']} outliers detected")
        print(f"     → Suggested clip at {row['Suggested Clip']:.2f} (current max: {row['Max']:.2f})")

print("\n" + display_df.to_string(index=False))
print("="*120)

# Save stats to CSV
stats_df.to_csv(f'{output_step3_path}/attributes_statistics.csv', index=False)
print(f"\n Statistics saved to: {output_step3_path}/attributes_statistics.csv")

# ============== GENERATE CLIPPING CODE ==============
print("\n" + "="*80)
print("📝 SUGGESTED CLIPPING CODE (choose threshold):")
print("="*80)

# For large value attributes, suggest Q75 or Q95
for attr in large_value_attrs:
    row = stats_df[stats_df['Attribute'] == attr].iloc[0]
    q75 = row['Q75']
    q95 = row['Q95']
    print(f"\n# {attr} (Large values):")
    print(f"# Option 1 (conservative - Q75): Indexv2['{attr}'] = Indexv2['{attr}'].clip(upper={q75:.2f})")
    print(f"# Option 2 (moderate - Q95):     Indexv2['{attr}'] = Indexv2['{attr}'].clip(upper={q95:.2f})")

# For outlier attributes
if needs_clipping:
    print("\n# Extreme outliers (automatic suggestion):")
    for attr in needs_clipping:
        if attr not in large_value_attrs:  # avoid duplicates
            clip_val = stats_df[stats_df['Attribute'] == attr]['Suggested Clip'].values[0]
            print(f"Indexv2['{attr}'] = Indexv2['{attr}'].clip(upper={clip_val:.2f})")

print("="*80)

**Normalize and center**

In [ ]:
# ============== CLIPPING BEFORE NORMALIZATION ==============
print("="*80)
print("✂️  APPLYING CLIPPING (based on attributs_info.clip column)")
print("="*80)

Indexv2 = Indexv1.copy()

for _, row in attributs_info.iterrows():
    attr = row['attribute']
    clip_value = row['clip']
    
    if attr in Indexv2.columns:
        if pd.notna(clip_value):
            original_max = Indexv2[attr].max()
            Indexv2[attr] = Indexv2[attr].clip(upper=clip_value)
            n_clipped = (Indexv1[attr] > clip_value).sum()
            print(f"✂️  {attr}: clipped at {clip_value:.2f} (was {original_max:.2f}, {n_clipped} values affected)")
        else:
            print(f"   {attr}: no clipping (clip=NaN)")
    else:
        print(f"⚠️  {attr}: not found in Indexv2")

print("="*80)

# ============== NORMALIZATION + OPTIONAL CENTERING ==============
CENTER_AROUND_ZERO = False  # Set to False for [0, 1] range and True for [-1, 1] range

Indexv3 = Indexv2.copy()
scaler = MinMaxScaler()

print("\n" + "="*80)
if CENTER_AROUND_ZERO:
    print("📊 MIN-MAX NORMALIZATION + CENTERING (range: [-1, 1])")
else:
    print("📊 MIN-MAX NORMALIZATION (range: [0, 1])")
print("="*80)

for attribute in attributs_info['attribute']:
    if attribute in Indexv3.columns:
        # Normalize to [0, 1]
        normalized = scaler.fit_transform(Indexv3[[attribute]])
        
        if CENTER_AROUND_ZERO:
            # Center to [-1, 1]: (x * 2) - 1
            Indexv3.loc[:, attribute] = (normalized * 2 - 1).round(4)
        else:
            # Keep [0, 1]
            Indexv3.loc[:, attribute] = normalized.round(4)
    else:
        print(f"⚠️  Attribute '{attribute}' not found in Indexv3 columns.")

print("✅ Normalization complete")

# ============== INVERSE COLUMNS (for defavorable attributes) ==============
print("\n" + "="*80)
print("🔄 INVERTING DEFAVORABLE ATTRIBUTES")
print("="*80)

for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    impact = row['impact_attribut']
    
    if attribute_name in Indexv3.columns:
        if impact == 'defavorable':
            if CENTER_AROUND_ZERO:
                # For [-1, 1]: multiply by -1
                Indexv3[attribute_name] = -Indexv3[attribute_name]
                print(f"🔄 Inverted {attribute_name}: multiplied by -1 (now favorable = positive)")
            else:
                # For [0, 1]: 1 - x
                Indexv3[attribute_name] = 1 - Indexv3[attribute_name]
                print(f"🔄 Inverted {attribute_name}: 1 - x")
    else:
        print(f"⚠️  Attribute '{attribute_name}' not found in Indexv3 columns.")

print("="*80)

# ============== SHOW DISTRIBUTION AFTER NORMALIZATION ==============
print("\n" + "="*80)
print("📊 POST-NORMALIZATION DISTRIBUTION CHECK")
print("="*80)

for attr in attributs_info['attribute'].head(5):  # Show first 5 as example
    if attr in Indexv3.columns:
        min_val = Indexv3[attr].min()
        max_val = Indexv3[attr].max()
        mean_val = Indexv3[attr].mean()
        impact = attributs_info[attributs_info['attribute'] == attr]['impact_attribut'].values[0]
        
        print(f"\n{attr} ({impact}):")
        print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
        print(f"  Mean:  {mean_val:.4f}")
        
        if CENTER_AROUND_ZERO:
            if impact == 'favorable':
                print(f"  ✅ Positive values = good walkability")
            else:
                print(f"  ✅ Negative values = bad walkability (inverted)")
        else:
            print(f"  ✅ Higher values = better walkability (after inversion if needed)")

print("="*80)

In [ ]:
# SIMPLE BOXPLOTS - All attributes on same axis with Class labels
import plotly.graph_objects as go
import pandas as pd
import numpy as np

attr_cols = [col for col in Indexv3.columns if col in attributes_to_include]

# Prepare data for boxplots
fig = go.Figure()

for attr in attr_cols:
    data = Indexv3[attr].dropna()
    impact = attributs_info[attributs_info['attribute'] == attr]['impact_attribut'].values[0]
    cls = attributs_info[attributs_info['attribute'] == attr]['Class'].values[0]
    
    # Color based on impact
    color = 'darkred' if impact == 'defavorable' else 'darkgreen'
    
    # Add boxplot
    fig.add_trace(go.Box(
        y=data,
        name=f"{cls}<br><b>{attr}</b><br>({impact})",
        boxmean='sd',
        marker_color=color,
        line=dict(width=1.2, color=color),
        fillcolor=color,
        opacity=0.6,
        showlegend=False,
        width=0.5
    ))

# Update layout
range_str = "[-1, 1]" if CENTER_AROUND_ZERO else "[0, 1]"
fig.update_layout(
    title=f"Normalized Attributes Distribution {range_str}",
    yaxis_title="Normalized Value",
    xaxis_title="Attributes",
    height=600,
    width=1400,
    template='plotly_white',
    showlegend=False,
    xaxis=dict(tickangle=-45, tickfont=dict(size=8))
)

# Add reference line at zero if centered
if CENTER_AROUND_ZERO:
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

# Save as PNG
# fig.write_image(f'{output_step3_path}/attributes_normalized_boxplots.png', width=1400, height=600, scale=2)
fig.show()

# Print summary statistics
print("\n" + "="*100)
print(f"DISTRIBUTION SUMMARY {range_str}")
print("="*100)

summary_data = []
for attr in attr_cols:
    data = Indexv3[attr].dropna()
    impact = attributs_info[attributs_info['attribute'] == attr]['impact_attribut'].values[0]
    cls = attributs_info[attributs_info['attribute'] == attr]['Class'].values[0]
    
    summary_data.append({
        'Class': cls,
        'Attribute': attr,
        'Impact': impact,
        'Min': data.min(),
        'Q25': data.quantile(0.25),
        'Median': data.median(),
        'Q75': data.quantile(0.75),
        'Max': data.max(),
        'Mean': data.mean(),
        'Std': data.std(),
        'IQR': data.quantile(0.75) - data.quantile(0.25)
    })

summary_df = pd.DataFrame(summary_data)

# Format for display
display_df = summary_df.copy()
for col in ['Min', 'Q25', 'Median', 'Q75', 'Max', 'Mean', 'Std', 'IQR']:
    display_df[col] = display_df[col].apply(lambda x: f'{x:.3f}')

print("\n" + display_df.to_string(index=False))
print("="*100)

# Check distribution quality
print("\nDISTRIBUTION QUALITY:")
for _, row in summary_df.iterrows():
    if row['Std'] < 0.1:
        print(f"  {row['Attribute']} ({row['Class']}): Low variance (std={row['Std']:.3f})")
    elif row['Std'] > 0.3:
        print(f"  {row['Attribute']} ({row['Class']}): Good spread (std={row['Std']:.3f})")
    else:
        print(f"  {row['Attribute']} ({row['Class']}): Moderate (std={row['Std']:.3f})")

print(f"\nPlot saved: {output_step3_path}/attributes_normalized_boxplots.png")

In [ ]:
# Calculate the Main Index Scores
Indexv4 = Indexv3.copy()

# Get weights from attributs_info and check validity
Zscore_weights = {}
for _, row in attributs_info[attributs_info.include_in_index == True].iterrows():
    attr = row['attribute']
    weight = row['initial_weight']
    if pd.isnull(weight):
        raise ValueError(f"Weight not specified for attribute '{attr}' (found NaN). Please specify a value between 0 and 1.")
    if not (0 <= weight <= 1):
        raise ValueError(f"Weight for attribute '{attr}' is {weight}, but must be between 0 and 1.")
    Zscore_weights[attr] = weight

print("Zscore_weights:", Zscore_weights)

# Add zscore weights to the attributes_info dataframe
attributs_info['weights'] = attributs_info['attribute'].map(Zscore_weights)

# Function Sum-product to calculate sub-index score
def calculate_index(df, weights_dict):
    return sum(df[col] * weight for col, weight in weights_dict.items())

# Calculate weighted and unweighted indices
Indexv4['walk_index'] = calculate_index(Indexv4, Zscore_weights)
Indexv4['walk_index_unweighted'] = calculate_index(Indexv4, {key: 1 for key in Zscore_weights.keys()})

# Display raw index range before normalization
print("\n" + "="*80)
print("📊 RAW INDEX RANGE (before normalization):")
print("="*80)
print(f"walk_index (weighted):     min={Indexv4['walk_index'].min():.4f}, max={Indexv4['walk_index'].max():.4f}, mean={Indexv4['walk_index'].mean():.4f}")
print(f"walk_index (unweighted):   min={Indexv4['walk_index_unweighted'].min():.4f}, max={Indexv4['walk_index_unweighted'].max():.4f}, mean={Indexv4['walk_index_unweighted'].mean():.4f}")

# Normalize indices based on CENTER_AROUND_ZERO parameter
index_scaler = MinMaxScaler()

if CENTER_AROUND_ZERO:
    print("\nℹ️  CENTER_AROUND_ZERO=True:")
    print("   - Attributes in [-1, 1]: negative=bad, positive=good")
    print("   - Index will be normalized to [-1, 1] for consistency")
    print("   - Interpretation: negative=bad walkability, positive=good walkability")
    
    # Normalize to [0, 1] first
    Indexv4[['walk_index']] = index_scaler.fit_transform(Indexv4[['walk_index']])
    Indexv4[['walk_index_unweighted']] = index_scaler.fit_transform(Indexv4[['walk_index_unweighted']])
    
    # Then center to [-1, 1]: (x * 2) - 1
    Indexv4['walk_index'] = (Indexv4['walk_index'] * 2 - 1).round(4)
    Indexv4['walk_index_unweighted'] = (Indexv4['walk_index_unweighted'] * 2 - 1).round(4)
    
    index_range_str = "[-1, 1]"
    
else:
    print("\nℹ️  CENTER_AROUND_ZERO=False:")
    print("   - Attributes in [0, 1]: 0=bad, 1=good")
    print("   - Index will be normalized to [0, 1] for consistency")
    print("   - Interpretation: 0=worst walkability, 1=best walkability")
    
    # Normalize to [0, 1]
    Indexv4[['walk_index']] = index_scaler.fit_transform(Indexv4[['walk_index']]).round(4)
    Indexv4[['walk_index_unweighted']] = index_scaler.fit_transform(Indexv4[['walk_index_unweighted']]).round(4)
    
    index_range_str = "[0, 1]"

print("="*80)

print(f"\n✅ NORMALIZED INDEX RANGE {index_range_str}:")
print(f"walk_index (weighted):     min={Indexv4['walk_index'].min():.4f}, max={Indexv4['walk_index'].max():.4f}, mean={Indexv4['walk_index'].mean():.4f}")
print(f"walk_index (unweighted):   min={Indexv4['walk_index_unweighted'].min():.4f}, max={Indexv4['walk_index_unweighted'].max():.4f}, mean={Indexv4['walk_index_unweighted'].mean():.4f}")
print("="*80)

**Appliquer un bonus aux zones piétonnes**

In [ ]:
# ============== PEDESTRIAN ZONE HANDLING ==============
print("\n" + "="*80)
print("🚶 PEDESTRIAN ZONE HANDLING")
print("="*80)

Indexv5 = Indexv4.copy()

# Check if zone_pietonne exists and has non-zero values
if 'zone_pietonne' in Indexv5.columns:
    n_pedestrian = (Indexv5["zone_pietonne"] > 0).sum()
    n_total = len(Indexv5)
    pct_pedestrian = (n_pedestrian / n_total) * 100
    
    if n_pedestrian > 0:
        print(f"ℹ️  Found {n_pedestrian:,} pedestrian zone segments ({pct_pedestrian:.2f}%)")
        
        # Option 1: NO OVERRIDE - Let the index be calculated normally
        # This respects other attributes (width, slope, obstacles, etc.)
        Indexv5["indice_marchabilite"] = Indexv5["walk_index"]
        
        # Analyze pedestrian zones
        ped_zones = Indexv5[Indexv5["zone_pietonne"] > 0]
        ped_index_stats = {
            'min': ped_zones["walk_index"].min(),
            'mean': ped_zones["walk_index"].mean(),
            'median': ped_zones["walk_index"].median(),
            'max': ped_zones["walk_index"].max()
        }
        
        print(f"\n📊 Pedestrian zone index statistics:")
        print(f"   Min:    {ped_index_stats['min']:.4f}")
        print(f"   Mean:   {ped_index_stats['mean']:.4f}")
        print(f"   Median: {ped_index_stats['median']:.4f}")
        print(f"   Max:    {ped_index_stats['max']:.4f}")
        
        # Option 2: BONUS APPROACH - Add a bonus instead of override
        # Uncomment to use this approach:
        """
        PEDESTRIAN_BONUS = 0.1  # Add 10% bonus in [0,1] or 0.2 in [-1,1]
        Indexv5["indice_marchabilite"] = np.where(
            Indexv5["zone_pietonne"] > 0,
            (Indexv5["walk_index"] + PEDESTRIAN_BONUS).clip(
                upper=1.0 if not CENTER_AROUND_ZERO else 1.0
            ),
            Indexv5["walk_index"]
        )
        print(f"\n✅ Applied +{PEDESTRIAN_BONUS} bonus to pedestrian zones")
        """
        
        # Option 3: PERCENTILE BOOST - Boost to high percentile
        # Uncomment to use this approach:
        """
        PERCENTILE_TARGET = 0.90  # Boost to 90th percentile
        target_value = Indexv5["walk_index"].quantile(PERCENTILE_TARGET)
        Indexv5["indice_marchabilite"] = np.where(
            (Indexv5["zone_pietonne"] > 0) & (Indexv5["walk_index"] < target_value),
            target_value,
            Indexv5["walk_index"]
        )
        print(f"\n✅ Boosted pedestrian zones below P{PERCENTILE_TARGET*100:.0f} to {target_value:.4f}")
        """
        
        # Option 4: MAXIMUM OVERRIDE (original approach) - Most aggressive
        # Uncomment to use this approach:
        """
        max_score = Indexv4["walk_index"].max()
        Indexv5["indice_marchabilite"] = np.where(
            Indexv5["zone_pietonne"] > 0,
            max_score,
            Indexv5["walk_index"]
        )
        print(f"\n⚠️  OVERRIDE: Set all {n_pedestrian:,} pedestrian zones to max score ({max_score:.4f})")
        """
        
        print("\n💡 Current approach: NO OVERRIDE")
        print("   → Pedestrian zones evaluated like other segments")
        print("   → To change: uncomment one of the alternative approaches above")
        
    else:
        print("ℹ️  No pedestrian zone segments found")
        Indexv5["indice_marchabilite"] = Indexv5["walk_index"]
else:
    print("⚠️  'zone_pietonne' column not found")
    Indexv5["indice_marchabilite"] = Indexv5["walk_index"]

print("="*80)

In [ ]:
# Calculate index for each class
Indexv6 = Indexv5.copy()  # 👈 IMPORTANT: Use .copy() to avoid SettingWithCopyWarning

print("\n" + "="*80)
print(f"📊 CALCULATING CLASS SUB-INDICES (will be in {index_range_str}):")
print("="*80)

for cls in attributs_info['Class'].unique():
    # Select attributes belonging to this class and included in the index
    subset = attributs_info[
        (attributs_info['Class'] == cls) & 
        (attributs_info['include_in_index'])
    ]

    # Build the weight dictionary for this class only
    cls_weights = {row['attribute']: row['class_weight'] for _, row in subset.iterrows()}

    if not cls_weights:
        print(f"⚠️  No attributes found for class '{cls}'")
        continue

    # Compute weighted sub-index (returns a Series)
    raw_values = calculate_index(Indexv6, cls_weights)
    
    # Normalize the result using separate scaler
    class_scaler = MinMaxScaler()
    normalized_values = class_scaler.fit_transform(raw_values.values.reshape(-1, 1))
    
    # Apply same transformation as main index for consistency
    if CENTER_AROUND_ZERO:
        # Center to [-1, 1] for consistency with attributes
        Indexv6[f"Classe_{cls}"] = ((normalized_values * 2 - 1).flatten()).round(4)
    else:
        # Keep [0, 1]
        Indexv6[f"Classe_{cls}"] = (normalized_values.flatten()).round(4)
    
    # Display stats
    col_name = f"Classe_{cls}"
    print(f"✅ {cls:20s}: min={Indexv6[col_name].min():.4f}, max={Indexv6[col_name].max():.4f}, mean={Indexv6[col_name].mean():.4f}")

print("="*80)

**Explore indexv6**

In [ ]:
Indexv6

**SAVE**

In [ ]:
# Save it

dirpath = output_step3_path
os.makedirs(dirpath, exist_ok=True)


Indexv6.to_crs(target_crs).to_csv(f'{output_step3_path}/step3_index.csv', index = False)
Indexv6.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_index.parquet')
Indexv6.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_index.gpkg"), driver="GPKG")